In [1]:
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from stripped_deepsdf_model import Model  # your minimal Model class




In [2]:
import torch
import numpy as np

from skimage import measure
import trimesh

# --- Device setup ---
device='mps'
print(f"Using device: {device}")

# --- Load model ---
model = Model().to(device)
ckpt_path = "sdf_weights/surfaces/cat-reference.ckpt"
ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt.get("state_dict", ckpt)
model.load_state_dict({k.replace("model.", ""): v for k, v in state_dict.items()}, strict=False)
model.eval()

# --- Inference ---
res =128  # you can increase for finer detail
with torch.no_grad():
    sdf = model.inference(v_res=res).cpu().numpy()  # shape = (res, res, res)

# --- Marching cubes ---
# Note: `level=0` extracts the zero level set (surface)
verts, faces, normals, values = measure.marching_cubes(sdf, level=0.0)

# Scale vertices from grid coordinates [-1,1] space
verts = verts / res * 2.0 - 1.0

# --- Visualize / export ---
mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals)
mesh.export("output_mesh.ply")
print("Mesh saved to output_mesh.ply")

# --- compute vertices and faces ---
verts, faces, normals, _ = measure.marching_cubes(sdf, level=0.0)

# Scale vertices from voxel indices to [-1,1] coordinates
verts = verts / sdf.shape[0] * 2.0 - 1.0

# Compute vertex normals in world space
mesh = trimesh.Trimesh(vertices=verts, faces=faces)
mesh.fix_normals()  # recomputes normals automatically

# Export and visualize
mesh.export("output/output_mesh.ply")
mesh.show()


Using device: mps


/Users/romywilliamson/miniconda3/envs/bens/lib/python3.8/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
/var/folders/3s/dfq9yqvd6y9303d_mb00rpxw0000gp/T/ipykernel_2715/3756730163.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this m

Mesh saved to output_mesh.ply
